# Zambia v2 — Google Earth Engine (GEE) scaffold


**What this is *not*:** the **Google Earth** phone/desktop app for spinning a globe. **Google Earth Engine** is a **separate** product: a **cloud catalog of raster datasets** + an API so Python (or the Code Editor) can query and summarize imagery **without downloading whole countries to your laptop first**.


## §0 — One-time setup (outside the notebook)

1. **Register** for Earth Engine access (Google account): open the [Earth Engine signup](https://code.earthengine.google.com/register) page and complete registration if you have not already.
2. **Google Cloud project:** create or pick a project in [Google Cloud Console](https://console.cloud.google.com/) and note its **project id** (e.g. `ipv-exposure-research`). Enable **Earth Engine API** for that project if prompted when you first use EE with it.
3. **Install the Python library** (repo venv): from the repository root, `pip install -r requirements.txt` (includes `earthengine-api`).
4. **Authenticate once on this computer** (terminal, not Jupyter):

   ```bash
   earthengine authenticate
   ```

   A browser window opens; finish the flow. Official guide: [Python install & auth](https://developers.google.com/earth-engine/guides/python_install).

5. **Tell Python which Cloud project to use:** set `EARTHENGINE_PROJECT` to your GCP **project id** (not the numeric project number). Options: (a) add it to the repo’s **`.env`** file at the repository root (gitignored); §1 loads that file automatically if `python-dotenv` is installed; (b) `export EARTHENGINE_PROJECT=...` before Jupyter or before `python -m gee_zambia.hansen_zonal` (the CLI reads this variable; see `gee_zambia/README.md`).


## §1 — Repo root (same idea as Zambia v1)

We need `ROOT` so we can read files the GEE script writes under `data/raw/...`. The next cell also loads **`ROOT/.env`** when `python-dotenv` is available, so variables like **`EARTHENGINE_PROJECT`** are visible to §2 without exporting them in the shell.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

_repo = next(
    (
        d
        for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (d / "utils" / "repo_paths.py").is_file() and (d / "data" / "raw").is_dir()
    ),
    None,
)
if _repo is None:
    raise RuntimeError("Cannot find repo root (expected utils/repo_paths.py and data/raw/).")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from utils.repo_paths import find_repo_root

ROOT = find_repo_root()
try:
    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env", override=False)
except ImportError:
    pass
print("ROOT =", ROOT)


## §2 — Does Earth Engine accept this session?

`ee.Initialize(project=...)` attaches your notebook kernel to your **authenticated** account **and** a Google Cloud **project id** Earth Engine can bill/register against. If this fails after `earthengine authenticate`, check the **project id** and that the **Earth Engine API** is enabled for that project.


In [ ]:
import os

import ee

# From shell or from ROOT/.env (loaded in §1). Value must be the GCP *project id*, not the numeric project number.
EE_PROJECT = (os.environ.get("EARTHENGINE_PROJECT") or "ipv-exposure-research").strip()

try:
    ee.Initialize(project=EE_PROJECT)
    print("OK: Earth Engine initialized (project=%s)." % EE_PROJECT)
except Exception as e:
    print("EE init failed:", type(e).__name__, e)
    print("Fix: (1) terminal: earthengine authenticate")
    print("      (2) set EARTHENGINE_PROJECT (e.g. in .env); enable Earth Engine API for that project")


## §3 — “Hello raster”: one elevation value (SRTM)

We load a **global digital elevation model** already hosted in the catalog (`USGS/SRTMGL1_003`), sample **one pixel** near Lusaka, and print the value. This proves: **catalog → image → sample → `getInfo()`** round-trip works.


In [ ]:
dem = ee.Image("USGS/SRTMGL1_003").select("elevation")
pt = ee.Geometry.Point(28.35, -15.42)  # rough Lusaka area; units are degrees lon/lat
sample_fc = dem.sample(region=pt, scale=30, numPixels=1, seed=0)
val = sample_fc.first().get("elevation").getInfo()
print("SRTM elevation (m) at sample point:", val)


## §4 — Hansen Global Forest Change (one-point “EDA”)

**Dataset (catalog):** [`UMD/hansen/global_forest_change_2025_v1_13`](https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2025_v1_13) (replaces deprecated `…2024_v1_12`).

**What you are looking at (three band names):**

| Band | Plain language |
|------|----------------|
| `treecover2000` | Model’s estimate of **% tree cover in 2000** at that pixel. |
| `lossyear` | **0** = no mapped loss; **1–25** = loss attributed mainly to calendar years **2001–2025** (encoding: year 2000 + value). So **2013 → 13**. This is **annual**, not “which month”. |
| `gain` | Forest **gain** flag (see dataset notes; not updated every year like loss). |

Below we print band names and **one dictionary** of values at the same sample point — that is **EDA** here: not statistics across Zambia yet, just “does the data read sensibly?” We use **`reduceRegion`** on a **`Point`** (not `sample().first()`, which can return a null feature for a zero-area region).


In [ ]:
hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")
print("Band names:", hansen.bandNames().getInfo()[:12], "...")

pt = ee.Geometry.Point(28.35, -15.42)
# reduceRegion reads the pixel under the point; sample().first() can be null for a zero-area region.
bands = ["treecover2000", "lossyear", "gain", "loss"]
props = (
    hansen.select(bands)
    .reduceRegion(reducer=ee.Reducer.first(), geometry=pt, scale=30, maxPixels=1e9)
    .getInfo()
)
for key in bands:
    if key in props and props[key] is not None:
        print(f"  {key}: {props[key]}")


## §5 — Optional: admin-2 zonal table (Hansen **2013 loss area** in hectares)

The heavy step (summarize **loss** pixels per **district polygon** for all of Zambia) lives in a **script** so this notebook stays short.

**From the repository root** (after §2 works):

```bash
python -m gee_zambia.hansen_zonal --year 2013
```

That writes `data/raw/exposure_gee/zambia/hansen_loss_y2013_admin2_zambia.csv`. If the run is slow or times out, try `--simplified` (coarser GAUL geometries). Details: `gee_zambia/README.md`.


In [ ]:
gee_csv = ROOT / "data" / "raw" / "exposure_gee" / "zambia" / "hansen_loss_y2013_admin2_zambia.csv"
if not gee_csv.is_file():
    print("No CSV yet — run: python -m gee_zambia.hansen_zonal --year 2013")
    print("Expected:", gee_csv)
else:
    df = pd.read_csv(gee_csv)
    print("rows:", len(df), "cols:", list(df.columns)[:8], "...")
    if "loss_area_ha" in df.columns:
        print("total loss_area_ha (sum over districts):", round(df["loss_area_ha"].sum(), 2))
    display(df.head(8))
